<a href="https://colab.research.google.com/github/kekubhai/Build-/blob/main/BasicCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("=" * 60)
print("CONVOLUTIONAL NEURAL NETWORK - LAYER IMPLEMENTATION")
print("=" * 60)

# ============================================================================
# 1. BASIC CONVOLUTION OPERATION
# ============================================================================
print("\n" + "=" * 60)
print("1. BASIC CONVOLUTION OPERATION")
print("=" * 60)

# Create a sample 6x6 image (like in the PDF example)
image_6x6 = torch.tensor([
    [3, 0, 1, 2, 7, 4],
    [1, 5, 8, 9, 3, 1],
    [2, 7, 2, 5, 1, 3],
    [0, 1, 3, 1, 7, 8],
    [4, 2, 1, 6, 2, 8],
    [2, 4, 5, 2, 3, 9]
], dtype=torch.float32)

print(f"Original 6x6 image:\n{image_6x6}")
print(f"Shape: {image_6x6.shape}")

# Reshape for PyTorch conv2d: (batch, channels, height, width)
image_tensor = image_6x6.unsqueeze(0).unsqueeze(0)  # (1, 1, 6, 6)
print(f"Tensor shape for conv2d: {image_tensor.shape}")

# Vertical edge detection filter (Sobel-like)
vertical_filter = torch.tensor([[
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
]], dtype=torch.float32).unsqueeze(0)  # (1, 1, 3, 3)

print(f"\nVertical edge detection filter (3x3):\n{vertical_filter.squeeze()}")

# Apply convolution with stride=1, no padding
conv_output = F.conv2d(image_tensor, vertical_filter, stride=1, padding=0)
print(f"\nConvolution result (4x4):\n{conv_output.squeeze()}")
print(f"Output shape: {conv_output.shape}")

# ============================================================================
# 2. EDGE DETECTION DEMONSTRATION
# ============================================================================
print("\n" + "=" * 60)
print("2. EDGE DETECTION DEMONSTRATION")
print("=" * 60)

# Create an image with clear vertical edge (light left, dark right)
image_vertical_edge = torch.zeros(6, 6)
image_vertical_edge[:, :3] = 10  # Left half bright
image_vertical_edge[:, 3:] = 0   # Right half dark
print(f"Image with vertical edge (light→dark):\n{image_vertical_edge}")

# Create an image with reverse vertical edge (dark left, light right)
image_vertical_edge_rev = torch.zeros(6, 6)
image_vertical_edge_rev[:, :3] = 0   # Left half dark
image_vertical_edge_rev[:, 3:] = 10  # Right half bright
print(f"\nImage with reverse vertical edge (dark→light):\n{image_vertical_edge_rev}")

# Create an image with horizontal edge (top bright, bottom dark)
image_horizontal_edge = torch.zeros(6, 6)
image_horizontal_edge[:3, :] = 10  # Top half bright
image_horizontal_edge[3:, :] = 0   # Bottom half dark
print(f"\nImage with horizontal edge (bright→dark):\n{image_horizontal_edge}")

# Define filters
vertical_filter = torch.tensor([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=torch.float32)
horizontal_filter = torch.tensor([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=torch.float32)

# Reshape for conv2d
img_v = image_vertical_edge.unsqueeze(0).unsqueeze(0)
img_v_rev = image_vertical_edge_rev.unsqueeze(0).unsqueeze(0)
img_h = image_horizontal_edge.unsqueeze(0).unsqueeze(0)
vf = vertical_filter.unsqueeze(0).unsqueeze(0)
hf = horizontal_filter.unsqueeze(0).unsqueeze(0)

# Apply convolutions
print("\n--- Vertical Edge Detection ---")
v_result = F.conv2d(img_v, vf, padding=0)
v_rev_result = F.conv2d(img_v_rev, vf, padding=0)
print(f"Light→Dark edge result:\n{v_result.squeeze()}")
print(f"Dark→Light edge result:\n{v_rev_result.squeeze()}")

print("\n--- Horizontal Edge Detection ---")
h_result = F.conv2d(img_h, hf, padding=0)
print(f"Horizontal edge result:\n{h_result.squeeze()}")

# ============================================================================
# 3. PADDING AND STRIDE
# ============================================================================
print("\n" + "=" * 60)
print("3. PADDING AND STRIDE")
print("=" * 60)

# Create a random 7x7 image
image_7x7 = torch.randn(1, 1, 7, 7)
print(f"Original 7x7 image shape: {image_7x7.shape}")

# No padding, stride=1
output_no_pad = F.conv2d(image_7x7, vertical_filter.unsqueeze(0).unsqueeze(0), stride=1, padding=0)
print(f"No padding, stride=1: {output_no_pad.shape}")

# Padding=1, stride=1
output_pad1 = F.conv2d(image_7x7, vertical_filter.unsqueeze(0).unsqueeze(0), stride=1, padding=1)
print(f"Padding=1, stride=1: {output_pad1.shape}")

# No padding, stride=2
output_stride2 = F.conv2d(image_7x7, vertical_filter.unsqueeze(0).unsqueeze(0), stride=2, padding=0)
print(f"No padding, stride=2: {output_stride2.shape}")

# Padding=1, stride=2
output_pad1_stride2 = F.conv2d(image_7x7, vertical_filter.unsqueeze(0).unsqueeze(0), stride=2, padding=1)
print(f"Padding=1, stride=2: {output_pad1_stride2.shape}")

# Formula verification
n, f, p, s = 7, 3, 0, 2
output_size = ((n + 2*p - f) // s) + 1
print(f"\nFormula check: (({n} + 2*{p} - {f})/{s}) + 1 = {output_size}")

# ============================================================================
# 4. RGB IMAGE CONVOLUTION (MULTIPLE CHANNELS)
# ============================================================================
print("\n" + "=" * 60)
print("4. RGB IMAGE CONVOLUTION (MULTIPLE CHANNELS)")
print("=" * 60)

# Create a random 6x6 RGB image (3 channels)
rgb_image = torch.randn(1, 3, 6, 6)  # (batch, channels, height, width)
print(f"RGB image shape: {rgb_image.shape}")

# Create a 3x3x3 filter (for RGB)
filter_3x3x3 = torch.randn(1, 3, 3, 3)  # (out_channels, in_channels, height, width)
print(f"RGB filter shape: {filter_3x3x3.shape}")

# Apply convolution
rgb_output = F.conv2d(rgb_image, filter_3x3x3, stride=1, padding=0)
print(f"RGB convolution output shape: {rgb_output.shape}")

# ============================================================================
# 5. MULTIPLE FILTERS (FEATURE DETECTORS)
# ============================================================================
print("\n" + "=" * 60)
print("5. MULTIPLE FILTERS (FEATURE DETECTORS)")
print("=" * 60)

# Create multiple filters (vertical, horizontal, diagonal)
filters = torch.zeros(3, 3, 3, 3)  # 3 filters, 3 channels each, 3x3

# Filter 0: Vertical edge
filters[0, :, :, 0] = 1
filters[0, :, :, 2] = -1

# Filter 1: Horizontal edge
filters[1, :, 0, :] = 1
filters[1, :, 2, :] = -1

# Filter 2: Diagonal edge (top-left to bottom-right)
for c in range(3):
    for i in range(3):
        filters[2, c, i, i] = 2
        filters[2, c, i, 2-i] = -1

print(f"Multiple filters shape: {filters.shape}")

# Apply all filters at once
multi_output = F.conv2d(rgb_image, filters, stride=1, padding=0)
print(f"Output with multiple filters shape: {multi_output.shape}")
print(f"This means: {multi_output.shape[2]}x{multi_output.shape[3]} output with {multi_output.shape[1]} feature maps")

# ============================================================================
# 6. POOLING LAYERS (MAX and AVERAGE)
# ============================================================================
print("\n" + "=" * 60)
print("6. POOLING LAYERS (MAX and AVERAGE)")
print("=" * 60)

# Create a feature map (like in PDF page 3)
feature_map = torch.tensor([
    [67, 341, 231, 673],
    [311, 441, 468, 569],
    [611, 404, 691, 579],
    [415, 387, 502, 438]
], dtype=torch.float32)

print(f"Original 4x4 feature map:\n{feature_map}")
feature_map_tensor = feature_map.unsqueeze(0).unsqueeze(0)  # (1, 1, 4, 4)

# Max pooling with 2x2 filter, stride=2
max_pool = F.max_pool2d(feature_map_tensor, kernel_size=2, stride=2)
print(f"\nMax pooling (2x2, stride=2):\n{max_pool.squeeze()}")

# Average pooling with 2x2 filter, stride=2
avg_pool = F.avg_pool2d(feature_map_tensor, kernel_size=2, stride=2)
print(f"\nAverage pooling (2x2, stride=2):\n{avg_pool.squeeze()}")

# Verify manually for first region (top-left 2x2)
top_left = feature_map[0:2, 0:2]
print(f"\nTop-left 2x2 region:\n{top_left}")
print(f"Max value: {top_left.max().item()}, Avg value: {top_left.mean().item():.1f}")

# ============================================================================
# 7. CUSTOM CNN LAYER IMPLEMENTATION
# ============================================================================
print("\n" + "=" * 60)
print("7. CUSTOM CNN LAYER IMPLEMENTATION")
print("=" * 60)

class SimpleConvLayer(nn.Module):
    """A simple convolutional layer with activation"""

    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0, activation='relu'):
        super(SimpleConvLayer, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)

        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'sigmoid':
            self.activation = nn.Sigmoid()
        elif activation == 'tanh':
            self.activation = nn.Tanh()
        else:
            self.activation = nn.Identity()

    def forward(self, x):
        return self.activation(self.conv(x))

# Create a sample input
sample_input = torch.randn(1, 3, 32, 32)  # 32x32 RGB image
print(f"Sample input shape: {sample_input.shape}")

# Create a conv layer
conv_layer = SimpleConvLayer(in_channels=3, out_channels=8, kernel_size=5, stride=1, padding=2)
output = conv_layer(sample_input)
print(f"Conv layer output shape: {output.shape}")

# Calculate number of parameters
params = sum(p.numel() for p in conv_layer.parameters())
print(f"Number of parameters in this layer: {params}")
print(f"Formula: (5*5*3 + 1) * 8 = {params}")

# ============================================================================
# 8. COMPLETE CNN ARCHITECTURE
# ============================================================================
print("\n" + "=" * 60)
print("8. COMPLETE CNN ARCHITECTURE (like in PDF page 36)")
print("=" * 60)

class SampleCNN(nn.Module):
    """A CNN architecture similar to the example in PDF page 36"""

    def __init__(self, num_classes=10):
        super(SampleCNN, self).__init__()

        # Layer 1: Conv + Pool
        self.conv1 = nn.Conv2d(3, 8, kernel_size=5, stride=1, padding=0)  # 32x32 -> 28x28
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 28x28 -> 14x14

        # Layer 2: Conv + Pool
        self.conv2 = nn.Conv2d(8, 16, kernel_size=5, stride=1, padding=0)  # 14x14 -> 10x10
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 10x10 -> 5x5

        # Fully connected layers
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, num_classes)

        self.relu = nn.ReLU()

    def forward(self, x):
        # Conv1 + ReLU + Pool1
        x = self.pool1(self.relu(self.conv1(x)))
        print(f"After conv1+pool1: {x.shape}")

        # Conv2 + ReLU + Pool2
        x = self.pool2(self.relu(self.conv2(x)))
        print(f"After conv2+pool2: {x.shape}")

        # Flatten
        x = x.view(x.size(0), -1)
        print(f"After flatten: {x.shape}")

        # Fully connected layers
        x = self.relu(self.fc1(x))
        print(f"After FC1: {x.shape}")

        x = self.relu(self.fc2(x))
        print(f"After FC2: {x.shape}")

        x = self.fc3(x)
        print(f"After FC3 (output): {x.shape}")

        return x

# Create model and test forward pass
model = SampleCNN(num_classes=10)
print("\nModel Architecture:")
print(model)

print("\nForward Pass:")
test_input = torch.randn(2, 3, 32, 32)  # Batch of 2 images
output = model(test_input)

# Calculate total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# ============================================================================
# 9. LEARNABLE FILTERS DEMONSTRATION
# ============================================================================
print("\n" + "=" * 60)
print("9. LEARNABLE FILTERS DEMONSTRATION")
print("=" * 60)

# Create a simple dataset for edge detection
def create_edge_dataset(n_samples=100, img_size=8):
    X = []
    y = []

    for _ in range(n_samples):
        # Randomly choose edge type: 0=vertical, 1=horizontal, 2=diagonal
        edge_type = np.random.randint(0, 3)
        img = np.zeros((img_size, img_size))

        if edge_type == 0:  # Vertical edge
            split = np.random.randint(2, img_size-2)
            img[:, :split] = np.random.uniform(0.5, 1.0)
            img[:, split:] = np.random.uniform(0, 0.5)
        elif edge_type == 1:  # Horizontal edge
            split = np.random.randint(2, img_size-2)
            img[:split, :] = np.random.uniform(0.5, 1.0)
            img[split:, :] = np.random.uniform(0, 0.5)
        else:  # Diagonal edge
            for i in range(img_size):
                for j in range(img_size):
                    if i + j < img_size:
                        img[i, j] = np.random.uniform(0.5, 1.0)
                    else:
                        img[i, j] = np.random.uniform(0, 0.5)

        X.append(img)
        y.append(edge_type)

    return np.array(X), np.array(y)

# Generate dataset
X, y = create_edge_dataset(n_samples=500, img_size=8)
X = X.reshape(-1, 1, 8, 8).astype(np.float32)  # Add channel dimension
y = y.astype(np.int64)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create datasets and dataloaders
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_dataset = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Input shape: {X_train[0].shape}")

# Define a simple CNN to learn edge detectors
class EdgeDetectorCNN(nn.Module):
    def __init__(self):
        super(EdgeDetectorCNN, self).__init__()
        # This single conv layer will learn to detect edges
        self.conv1 = nn.Conv2d(1, 8, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(8 * 4 * 4, 3)  # 8x4x4 after pooling
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x

# Train the model
model = EdgeDetectorCNN()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("\nTraining CNN to learn edge detectors...")
epochs = 10
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    train_acc = 100. * correct / total

    # Validation
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    test_acc = 100. * correct / total

    print(f"Epoch {epoch+1}: Loss={running_loss/len(train_loader):.4f}, "
          f"Train Acc={train_acc:.2f}%, Test Acc={test_acc:.2f}%")

# Visualize learned filters
print("\nLearned filters (first 4 of 8):")
filters = model.conv1.weight.data
for i in range(min(4, filters.shape[0])):
    print(f"Filter {i}:\n{filters[i, 0, :, :].numpy().round(2)}")

# ============================================================================
# 10. PARAMETER COUNT DEMONSTRATION
# ============================================================================
print("\n" + "=" * 60)
print("10. PARAMETER COUNT DEMONSTRATION")
print("=" * 60)

def compare_parameters():
    """Compare parameters between fully connected and convolutional layers"""

    # Input: 32x32 RGB image
    input_size = 32 * 32 * 3

    # Fully connected layer with 1000 neurons
    fc_params = input_size * 1000 + 1000  # weights + bias
    print(f"Fully Connected (32x32x3 → 1000): {fc_params:,} parameters")

    # Convolutional layer with 8 filters of size 5x5
    conv_params = (5 * 5 * 3 + 1) * 8  # (kernel_size * in_channels + bias) * out_channels
    print(f"Convolutional (5x5 filters, 8 outputs): {conv_params} parameters")

    # For a 1000x1000 RGB image
    large_input_size = 1000 * 1000 * 3
    large_fc_params = large_input_size * 1000 + 1000
    print(f"\nFor 1000x1000x3 image:")
    print(f"FC layer: {large_fc_params:,} parameters (~{large_fc_params/1e9:.2f} billion)")

    # Same conv layer
    print(f"Conv layer: {conv_params} parameters (same!)")

compare_parameters()

# ============================================================================
# 11. POOLING VISUALIZATION
# ============================================================================
print("\n" + "=" * 60)
print("11. POOLING VISUALIZATION")
print("=" * 60)

# Create a feature map with patterns
feature_map = torch.zeros(1, 1, 8, 8)
feature_map[0, 0, 2:6, 2:6] = 1.0  # Square in the middle
feature_map[0, 0, 4, 4] = 2.0  # Peak in the center

print(f"Feature map with pattern:\n{feature_map.squeeze().numpy().round(2)}")

# Apply different pooling operations
max_pool = F.max_pool2d(feature_map, kernel_size=2, stride=2)
avg_pool = F.avg_pool2d(feature_map, kernel_size=2, stride=2)

print(f"\nMax pooling (preserves peaks):\n{max_pool.squeeze().numpy().round(2)}")
print(f"Average pooling (smooths):\n{avg_pool.squeeze().numpy().round(2)}")

# ============================================================================
# 12. CUSTOM IMPLEMENTATION OF CONV2D (for understanding)
# ============================================================================
print("\n" + "=" * 60)
print("12. CUSTOM CONVOLUTION IMPLEMENTATION")
print("=" * 60)

def custom_conv2d(image, kernel, stride=1, padding=0):
    """
    Manual 2D convolution implementation
    image: (H, W) or (C, H, W)
    kernel: (out_channels, in_channels, kH, kW) or (kH, kW)
    """
    # Handle single channel case
    if image.dim() == 2:
        image = image.unsqueeze(0)  # Add channel dimension
    if kernel.dim() == 2:
        kernel = kernel.unsqueeze(0).unsqueeze(0)  # Add out_channel and in_channel

    C, H, W = image.shape
    out_channels, in_channels, kH, kW = kernel.shape

    assert C == in_channels, f"Image channels ({C}) must match kernel in_channels ({in_channels})"

    # Apply padding
    if padding > 0:
        padded = torch.zeros(C, H + 2*padding, W + 2*padding)
        padded[:, padding:padding+H, padding:padding+W] = image
        H, W = H + 2*padding, W + 2*padding
    else:
        padded = image

    # Calculate output dimensions
    out_H = ((H - kH) // stride) + 1
    out_W = ((W - kW) // stride) + 1

    # Initialize output
    output = torch.zeros(out_channels, out_H, out_W)

    # Perform convolution
    for oc in range(out_channels):
        for i in range(0, out_H):
            for j in range(0, out_W):
                h_start = i * stride
                h_end = h_start + kH
                w_start = j * stride
                w_end = w_start + kW

                # Extract region and compute sum of products
                region = padded[:, h_start:h_end, w_start:w_end]
                output[oc, i, j] = torch.sum(region * kernel[oc])

    return output

# Test custom convolution
test_image = torch.tensor([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16]
], dtype=torch.float32)

test_kernel = torch.tensor([
    [1, 0, -1],
    [1, 0, -1],
    [1, 0, -1]
], dtype=torch.float32)

print(f"Test image:\n{test_image}")
print(f"Test kernel:\n{test_kernel}")

custom_result = custom_conv2d(test_image, test_kernel, stride=1, padding=0)
print(f"\nCustom convolution result:\n{custom_result.squeeze()}")

# Compare with PyTorch
pytorch_result = F.conv2d(
    test_image.unsqueeze(0).unsqueeze(0),
    test_kernel.unsqueeze(0).unsqueeze(0)
)
print(f"PyTorch convolution result:\n{pytorch_result.squeeze()}")

print(f"\nResults match: {torch.allclose(custom_result.squeeze(), pytorch_result.squeeze())}")

print("\n" + "=" * 60)
print("All demonstrations complete!")
print("=" * 60)

CONVOLUTIONAL NEURAL NETWORK - LAYER IMPLEMENTATIONS

1. BASIC CONVOLUTION OPERATION
Original 6x6 image:
tensor([[3., 0., 1., 2., 7., 4.],
        [1., 5., 8., 9., 3., 1.],
        [2., 7., 2., 5., 1., 3.],
        [0., 1., 3., 1., 7., 8.],
        [4., 2., 1., 6., 2., 8.],
        [2., 4., 5., 2., 3., 9.]])
Shape: torch.Size([6, 6])
Tensor shape for conv2d: torch.Size([1, 1, 6, 6])

Vertical edge detection filter (3x3):
tensor([[ 1.,  0., -1.],
        [ 1.,  0., -1.],
        [ 1.,  0., -1.]])

Convolution result (4x4):
tensor([[ -5.,  -4.,   0.,   8.],
        [-10.,  -2.,   2.,   3.],
        [  0.,  -2.,  -4.,  -7.],
        [ -3.,  -2.,  -3., -16.]])
Output shape: torch.Size([1, 1, 4, 4])

2. EDGE DETECTION DEMONSTRATION
Image with vertical edge (light→dark):
tensor([[10., 10., 10.,  0.,  0.,  0.],
        [10., 10., 10.,  0.,  0.,  0.],
        [10., 10., 10.,  0.,  0.,  0.],
        [10., 10., 10.,  0.,  0.,  0.],
        [10., 10., 10.,  0.,  0.,  0.],
        [10., 10., 10., 

In [ ]:
print("\n" + "=" * 60)
print("2. EDGE DETECTION DEMONSTRATION")
print("=" * 60)

# Create an image with clear vertical edge (light left, dark right)
image_vertical_edge = torch.zeros(6, 6)
image_vertical_edge[:, :3] = 10  # Left half bright
image_vertical_edge[:, 3:] = 0   # Right half dark
print(f"Image with vertical edge (light→dark):\n{image_vertical_edge}")



2. EDGE DETECTION DEMONSTRATION
Image with vertical edge (light→dark):
tensor([[10., 10., 10.,  0.,  0.,  0.],
        [10., 10., 10.,  0.,  0.,  0.],
        [10., 10., 10.,  0.,  0.,  0.],
        [10., 10., 10.,  0.,  0.,  0.],
        [10., 10., 10.,  0.,  0.,  0.],
        [10., 10., 10.,  0.,  0.,  0.]])
